# Demo: Decoder-Only Transformer, From Scratch

This notebook is a thin walkthrough on top of the actual project code
(`model.py`, `dataset.py`, `generate.py`, `visualize_attention.py`) it
doesn't reimplement anything, it just imports the real modules so you
can see the trained model in action without running scripts from a
terminal.

See `README.md` for the architecture, design rationale, and full results.

In [ ]:
import sys
sys.path.append("..")  # so we can import the project's .py modules from notebooks/

import torch
import matplotlib.pyplot as plt
from IPython.display import Image, display

from dataset import EOS, FACTS
from generate import load_model, generate, check_all_facts
from visualize_attention import get_attention, plot_attention

## 1. Load the trained model

In [ ]:
model, token_to_id, id_to_token = load_model("../checkpoints/model.pt")
print(f"vocab size: {len(token_to_id)}")
print(f"training facts: {len(FACTS)}")

## 2. Ask it some questions

Both directions (forward and backward) for a few facts, including ones
the model has to get right by attending to the right token, not by
memorizing a fixed position.

In [ ]:
prompts = [
    ["capital", "of", "japan", "?"],
    ["capital", "of", "kenya", "?"],
    ["nairobi", "is", "the", "capital", "of", "?"],
    ["capital", "of", "chile", "?"],
]

for prompt in prompts:
    answer = generate(model, token_to_id, id_to_token, prompt)
    print(f'{" ".join(prompt):45s} -> {" ".join(answer)}')

## 3. Full accuracy check

Re-runs every fact, both directions, 5 sampling trials each (not greedy
decoding — see `generate.py` for why that matters). This is where the
120/120 number in the README comes from.

In [ ]:
correct, total = check_all_facts(model, token_to_id, id_to_token, trials=5)
print(f"\n{correct}/{total} correct")

## 4. Where is the model actually looking?

The heatmap below is the real evidence the model is using attention
rather than memorizing slot positions: for the prompt "capital of kenya
? nairobi", the answer token should attend back to "kenya", not to a
fixed position — and it does, and it does this differently depending on
which country appears, which a position-memorizing model couldn't do.

In [ ]:
example_tokens = ["capital", "of", "kenya", "?", EOS, "nairobi"]
attn_weights = get_attention(model, token_to_id, example_tokens)
plot_attention(attn_weights, example_tokens, out_path="../assets/attention_heatmap.png")

display(Image("../assets/attention_heatmap.png"))

## 5. Try your own prompt

Anything using the vocabulary the model was trained on (see `dataset.py`
for the full list of countries/capitals).

In [ ]:
my_prompt = ["capital", "of", "germany", "?"]  # edit me
answer = generate(model, token_to_id, id_to_token, my_prompt)
print(" ".join(my_prompt), "->", " ".join(answer))